In [4]:
!python3 -m pip install pandas

Defaulting to user installation because normal site-packages is not writeable
     |████████████████████████████████| 10.8 MB 3.6 MB/s eta 0:00:01
     |████████████████████████████████| 510 kB 3.7 MB/s eta 0:00:01
     |████████████████████████████████| 349 kB 4.5 MB/s eta 0:00:01
     |████████████████████████████████| 5.3 MB 5.0 MB/s eta 0:00:01
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.


In [5]:
import pandas as pd
import json

In [9]:
ledger = pd.read_csv("../01_data/ledger.csv")
gateway = pd.read_csv("../01_data/gateway.csv")

print("Ledger loaded successfully")
print("Gateway loaded successfully")

Ledger loaded successfully
Gateway loaded successfully


In [10]:
print("Ledger duplicates:", ledger.duplicated().sum())
print("Gateway duplicates:", gateway.duplicated().sum())

Ledger duplicates: 0
Gateway duplicates: 0


In [11]:
print("Ledger null values:")
print(ledger.isnull().sum())

print("\nGateway null values:")
print(gateway.isnull().sum())

Ledger null values:
transaction_id      0
transaction_date    0
merchant_id         0
amount_usd          0
status              0
payment_method      0
dtype: int64

Gateway null values:
transaction_id      0
transaction_date    0
merchant_id         0
amount_usd          0
status              0
payment_method      0
dtype: int64


In [12]:
missing_in_gateway = ledger[
    ~ledger["transaction_id"].isin(gateway["transaction_id"])
]

print(missing_in_gateway)

  transaction_id transaction_date merchant_id  amount_usd   status  \
3           R004       2026-03-02        M003      2100.0  success   
9           R010       2026-03-05        M004      2500.0  success   

  payment_method  
3           Card  
9         Wallet  


In [15]:
missing_in_gateway.to_csv(
    "../01_data/processed/missing_in_gateway.csv",
    index=False
)

print("missing_in_gateway.csv saved successfully")

missing_in_gateway.csv saved successfully


In [16]:
missing_in_ledger = gateway[
    ~gateway["transaction_id"].isin(ledger["transaction_id"])
]

print(missing_in_ledger)

  transaction_id transaction_date merchant_id  amount_usd   status  \
8           R011       2026-03-05        M003      1800.0  success   

  payment_method  
8           Card  


In [17]:
missing_in_ledger.to_csv(
    "../01_data/processed/missing_in_ledger.csv",
    index=False
)

print("missing_in_ledger.csv saved successfully")

missing_in_ledger.csv saved successfully


In [4]:
merged_data = pd.merge(
    ledger,
    gateway,
    on="transaction_id",
    suffixes=("_ledger", "_gateway")
)

amount_mismatches = merged_data[
    merged_data["amount_usd_ledger"] != merged_data["amount_usd_gateway"]
]

print(amount_mismatches[[
    "transaction_id",
    "amount_usd_ledger",
    "amount_usd_gateway"
]])

  transaction_id  amount_usd_ledger  amount_usd_gateway
1           R002              850.0               900.0
6           R008              640.0               600.0


In [5]:
amount_mismatches.to_csv(
    "../01_data/processed/amount_mismatches.csv",
    index=False
)

print("amount_mismatches.csv saved successfully")

amount_mismatches.csv saved successfully


In [6]:
status_mismatches = merged_data[
    merged_data["status_ledger"] != merged_data["status_gateway"]
]

print(status_mismatches[[
    "transaction_id",
    "status_ledger",
    "status_gateway"
]])

  transaction_id status_ledger status_gateway
3           R005       success         failed


In [7]:
status_mismatches.to_csv(
    "../01_data/processed/status_mismatches.csv",
    index=False
)

print("status_mismatches.csv saved successfully")

status_mismatches.csv saved successfully


In [8]:
reconciliation_report = merged_data.copy()

reconciliation_report["reconciliation_status"] = "Matched"

# Mark amount mismatches
reconciliation_report.loc[
    reconciliation_report["amount_usd_ledger"] != reconciliation_report["amount_usd_gateway"],
    "reconciliation_status"
] = "Amount Mismatch"

# Mark status mismatches
reconciliation_report.loc[
    reconciliation_report["status_ledger"] != reconciliation_report["status_gateway"],
    "reconciliation_status"
] = "Status Mismatch"

print(reconciliation_report[[
    "transaction_id",
    "reconciliation_status"
]])

  transaction_id reconciliation_status
0           R001               Matched
1           R002       Amount Mismatch
2           R003               Matched
3           R005       Status Mismatch
4           R006               Matched
5           R007               Matched
6           R008       Amount Mismatch
7           R009               Matched


In [9]:
reconciliation_report.to_csv(
    "../01_data/processed/reconciliation_report.csv",
    index=False
)

print("reconciliation_report.csv saved successfully")

reconciliation_report.csv saved successfully


In [12]:
summary_metrics = {
    "missing_in_gateway": len(missing_in_gateway),
    "missing_in_ledger": len(missing_in_ledger),
    "amount_mismatches": len(amount_mismatches),
    "status_mismatches": len(status_mismatches),
    "total_ledger_records": len(ledger),
    "total_gateway_records": len(gateway)
}

print(summary_metrics)

{'missing_in_gateway': 2, 'missing_in_ledger': 1, 'amount_mismatches': 2, 'status_mismatches': 1, 'total_ledger_records': 10, 'total_gateway_records': 9}


In [13]:
with open("../04_python/summary_metrics.json", "w") as f:
    json.dump(summary_metrics, f, indent=4)

print("summary_metrics.json saved successfully")

summary_metrics.json saved successfully


In [14]:
import json
import pandas as pd

In [15]:
with open("../01_data/api_response_sample.json", "r") as f:
    data = json.load(f)

print(data)

{'generated_at': '2026-03-07T10:00:00Z', 'source': 'QuickPay Settlement API', 'batches': [{'batch_id': 'B001', 'merchant': {'merchant_id': 'M001', 'merchant_name': 'Alpha Mart', 'region': 'APAC'}, 'settlements': [{'settlement_id': 'S001', 'amount_usd': 1520.5, 'status': 'settled', 'processed_at': '2026-03-07T08:10:00Z', 'bank': {'name': 'Bank A', 'country': 'IN'}}, {'settlement_id': 'S002', 'amount_usd': 980.0, 'status': 'pending', 'processed_at': '2026-03-07T08:45:00Z', 'bank': {'name': 'Bank A', 'country': 'IN'}}, {'settlement_id': 'S003', 'amount_usd': 640.0, 'status': 'settled', 'processed_at': '2026-03-07T09:15:00Z', 'bank': {'name': 'Bank B', 'country': 'SG'}}]}, {'batch_id': 'B002', 'merchant': {'merchant_id': 'M004', 'merchant_name': 'Delta Travels', 'region': 'US'}, 'settlements': [{'settlement_id': 'S004', 'amount_usd': 2100.0, 'status': 'settled', 'processed_at': '2026-03-07T08:20:00Z', 'bank': {'name': 'Bank C', 'country': 'US'}}, {'settlement_id': 'S005', 'amount_usd': 500

In [16]:
api_df = pd.json_normalize(data)

api_df.head()

,generated_at,source,batches
0,2026-03-07T10:00:00Z,QuickPay Settlement API,"[{'batch_id': 'B001', 'merchant': {'merchant_i..."


In [17]:
api_df.columns = api_df.columns.str.replace(".", "_", regex=False)
api_df.columns = api_df.columns.str.lower()

print(api_df.columns)

Index(['generated_at', 'source', 'batches'], dtype='object')


In [18]:
exploded_df = api_df.explode("batches")

normalized_batches = pd.json_normalize(exploded_df["batches"])

final_api_df = pd.concat(
    [exploded_df.drop(columns=["batches"]).reset_index(drop=True),
     normalized_batches.reset_index(drop=True)],
    axis=1
)

final_api_df.head()

,generated_at,source,batch_id,settlements,merchant.merchant_id,merchant.merchant_name,merchant.region
0,2026-03-07T10:00:00Z,QuickPay Settlement API,B001,"[{'settlement_id': 'S001', 'amount_usd': 1520....",M001,Alpha Mart,APAC
1,2026-03-07T10:00:00Z,QuickPay Settlement API,B002,"[{'settlement_id': 'S004', 'amount_usd': 2100....",M004,Delta Travels,US


In [19]:
final_api_df.columns = final_api_df.columns.str.replace(".", "_", regex=False)
final_api_df.columns = final_api_df.columns.str.lower()

print(final_api_df.columns)

Index(['generated_at', 'source', 'batch_id', 'settlements',
       'merchant_merchant_id', 'merchant_merchant_name', 'merchant_region'],
      dtype='object')


In [20]:
final_api_df["generated_at"] = pd.to_datetime(final_api_df["generated_at"])

In [21]:
final_api_df.to_csv("../01_data/processed/api_normalized.csv", index=False)

print("api_normalized.csv saved successfully")

api_normalized.csv saved successfully


In [11]:
missing_in_gateway = ledger[
    ~ledger["transaction_id"].isin(gateway["transaction_id"])
]

missing_in_ledger = gateway[
    ~gateway["transaction_id"].isin(ledger["transaction_id"])
]

print("Variables recreated successfully")

Variables recreated successfully


In [2]:
import pandas as pd
import json
import os

In [3]:
ledger = pd.read_csv("../01_data/ledger.csv")
gateway = pd.read_csv("../01_data/gateway.csv")

print("Files loaded successfully")

Files loaded successfully


In [14]:
import os

os.makedirs("../01_data/processed", exist_ok=True)

print("processed folder created successfully")

processed folder created successfully
